# Add Your Own Method

Follow this tutorial to add a new clustering method to the benchmark.

## 1. Interface

Every method subclasses `FQAlgorithm` and implements `step(Xr, seed)`.

In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "fq").exists():
        sys.path.insert(0, str(candidate))
        break
from fq.algorithms.base import FQAlgorithm
import numpy as np, time

## 2. Worked Example

Random centroid reassignment samples centers and assigns points to their nearest sampled center.

In [ ]:
from fq.core import assign_labels, centroids_from_labels, compute_sse

class RandomReassign(FQAlgorithm):
    name = "RR"
    heavy = False
    def step(self, Xr, seed):
        t0 = time.perf_counter(); rng = np.random.default_rng(seed)
        centers0 = Xr[rng.choice(Xr.shape[0], self.K, replace=False)]
        labels = assign_labels(Xr, centers0)
        centers = centroids_from_labels(Xr, labels, self.K, rng)
        labels = assign_labels(Xr, centers)
        return dict(time=time.perf_counter()-t0, sse=compute_sse(Xr, centers, labels), iter=1, centers=centers)

## 3. Run Through the Benchmark

In [ ]:
from fq.data import synthesize_paper_data
from fq.runner import run_algorithm
X = synthesize_paper_data(nsim=200, R=128, seed=0)
rr = run_algorithm(RandomReassign(10, 5, 1e-3), [128], X, n_exp=3, verbose_every=0)
rr["name"], rr["all_sse"]

## 4. Register Permanently

Add `from .my_method import MyMethod` and `ALGORITHMS["MY"] = MyMethod` in `fq/algorithms/__init__.py`, then compare RD and time plots against LX.